# 03 — Model Training & Evaluation
### AI-Generated Bengali Handwritten Signature Detection

Trains and compares three models on the preprocessed, writer-split dataset produced by
Notebook 02 (`dataset/metadata_processed.csv` + `dataset/normalization_stats.csv` +
`processed_dataset/`):

1. **Custom CNN** (trained from scratch)
2. **ResNet18** (ImageNet-pretrained, transfer learning)
3. **EfficientNet-B0** (ImageNet-pretrained, transfer learning)

Then covers:
- Full evaluation (accuracy, precision, recall, F1, ROC-AUC, confusion matrix) — overall and
  broken down **per generator** (Genuine / Gemini / QN / GPT — whichever are present; this
  notebook doesn't hardcode which generators exist, so adding QN data later needs no edits)
- A **cross-generator generalization** experiment (train holding one generator out, test on it)
- **Grad-CAM** explainability on the best-performing model

**Self-contained:** this version defines `SignatureDataset` / transforms / metadata loading
directly in this notebook rather than importing a `data_utils.py` that was never produced by
Notebook 02 — this is a fix from the previous version, which would fail with
`ModuleNotFoundError: No module named 'data_utils'`.

**Hardware:** written for a 2GB-VRAM laptop GPU (MX350) per the project's hardware notes —
`batch_size=8`, `num_workers=0` (2 previously caused DataLoader issues on Windows/Jupyter).

## 1. Setup

In [2]:
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve)

warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "white"

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATASET_ROOT = Path("../dataset")
CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

# MX350 / 2GB VRAM settings -- raise BATCH_SIZE if you have more headroom, but keep
# num_workers at 0 (num_workers=2 previously caused DataLoader problems in this project).
BATCH_SIZE = 8
NUM_WORKERS = 0


Using device: cpu


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

PyTorch: 2.5.1+cu118
CUDA available: False
CUDA version: 11.8
GPU count: 0


## 2. Data loading

Reads the outputs of Notebook 02 directly — no external module required. `SignatureDataset`
accepts an optional `generators` filter (a list of generator names, e.g. `["Human","Gemini"]`)
so the same class covers the overall experiment and the per-generator / cross-generator ones
below without duplicating loading logic.

In [4]:
class SignatureDataset(Dataset):
    '''Reads preprocessed (RGB, 224x224) images listed in metadata_processed.csv.

    Parameters
    ----------
    metadata_df : DataFrame with at least [filepath, label, generator, split] columns.
    split : "train" | "val" | "test"
    transform : torchvision transform applied to the loaded PIL image.
    generators : optional list restricting rows to specific generator(s), e.g.
        ["Human", "Gemini"] for a Human-vs-Gemini-only experiment (master note Experiment 2).
        "Human" is always the genuine/label-0 rows in this project's metadata convention.
    '''

    def __init__(self, metadata_df, split, transform=None, generators=None):
        d = metadata_df[metadata_df["split"] == split]
        if generators is not None:
            d = d[d["generator"].isin(generators)]
        self.df = d.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"])


def load_metadata(dataset_root: Path):
    meta_path = dataset_root / "metadata_processed.csv"
    norm_path = dataset_root / "normalization_stats.csv"
    assert meta_path.exists(), f"{meta_path} not found -- run Notebook 02 first."
    assert norm_path.exists(), f"{norm_path} not found -- run Notebook 02 first."

    df = pd.read_csv(meta_path)
    norm = pd.read_csv(norm_path).iloc[0]
    mean = [norm["mean_r"], norm["mean_g"], norm["mean_b"]]
    std = [norm["std_r"], norm["std_g"], norm["std_b"]]
    return df, mean, std


def build_transforms(mean, std):
    train_tf = T.Compose([
        T.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.97, 1.03), fill=255),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    eval_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])
    return train_tf, eval_tf


metadata_df, NORM_MEAN, NORM_STD = load_metadata(DATASET_ROOT)
train_transform, eval_transform = build_transforms(NORM_MEAN, NORM_STD)

GENERATORS_PRESENT = sorted(metadata_df["generator"].unique().tolist())
print(f"Generators present in metadata: {GENERATORS_PRESENT}")

train_ds = SignatureDataset(metadata_df, "train", transform=train_transform)
val_ds = SignatureDataset(metadata_df, "val", transform=eval_transform)
test_ds = SignatureDataset(metadata_df, "test", transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

# Class weights -- computed from whatever the actual train-split counts are (don't hardcode
# a specific genuine/AI ratio, since dataset counts change as generators are added).
train_labels = metadata_df.loc[metadata_df["split"] == "train", "label"].values
class_counts = np.bincount(train_labels, minlength=2)
class_weights = torch.tensor(len(train_labels) / (2.0 * class_counts), dtype=torch.float32).to(DEVICE)
print(f"Train class counts: {class_counts.tolist()}, class weights: {class_weights.tolist()}")


Generators present in metadata: ['GPT', 'Gemini', 'Human', 'QN']
train: 1461, val: 308, test: 265
Train class counts: [695, 766], class weights: [1.0510791540145874, 0.9536553621292114]


## 3. Model definitions

In [5]:
class CustomCNN(nn.Module):
    '''Lightweight from-scratch baseline -- establishes whether the dataset carries enough
    visual signal before relying on transfer learning.'''

    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),    # 224->112
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),   # 112->56
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), # 56->28
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),# 28->14
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_resnet18(num_classes=2, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_efficientnet_b0(num_classes=2, freeze_backbone=True):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


MODEL_BUILDERS = {
    "custom_cnn": lambda: CustomCNN(),
    "resnet18": lambda: build_resnet18(freeze_backbone=True),
    "efficientnet_b0": lambda: build_efficientnet_b0(freeze_backbone=True),
}


### Training curves

In [7]:
checkpoint_files = list(CKPT_DIR.glob("*"))

for f in checkpoint_files:
    print(f.name)

custom_cnn_best.pt
custom_cnn_latest.pt
efficientnet_b0_best.pt
efficientnet_b0_latest.pt
resnet18_best.pt
resnet18_latest.pt


In [9]:
test_df = metadata_df[
    metadata_df["split"] == "test"
].copy()

print("Total test samples:", len(test_df))

display(
    test_df["generator"]
    .value_counts()
    .sort_index()
)

Total test samples: 265


generator
GPT        29
Gemini     27
Human     149
QN         60
Name: count, dtype: int64

In [11]:
trained_models = {}

for model_name in ["custom_cnn", "resnet18", "efficientnet_b0"]:

    ckpt_path = CKPT_DIR / f"{model_name}_best.pt"

    print(f"Loading {model_name} from:")
    print(ckpt_path)

    model = MODEL_BUILDERS[model_name]().to(DEVICE)

    checkpoint = torch.load(
        ckpt_path,
        map_location=DEVICE
    )

    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    trained_models[model_name] = model

    print(f"{model_name} loaded successfully.")

Loading custom_cnn from:
checkpoints\custom_cnn_best.pt
custom_cnn loaded successfully.
Loading resnet18 from:
checkpoints\resnet18_best.pt
resnet18 loaded successfully.
Loading efficientnet_b0 from:
checkpoints\efficientnet_b0_best.pt
efficientnet_b0 loaded successfully.


In [13]:
def evaluate_model(model, loader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.numpy())

    all_probs, all_preds, all_labels = np.array(all_probs), np.array(all_preds), np.array(all_labels)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds, labels=[0, 1]).ravel()

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
        "roc_auc": roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else float("nan"),
        "human_recall": tn / (tn + fp) if (tn + fp) > 0 else float("nan"),
        "ai_recall": tp / (tp + fn) if (tp + fn) > 0 else float("nan"),
    }
    return metrics, all_preds, all_probs, all_labels


results_summary = []
for name, model in trained_models.items():
    metrics, preds, probs, labels = evaluate_model(model, test_loader)
    metrics["model"] = name
    results_summary.append(metrics)

results_df = pd.DataFrame(results_summary).set_index("model")[
    ["accuracy", "precision", "recall", "f1", "roc_auc", "human_recall", "ai_recall"]]
results_df


,accuracy,precision,recall,f1,roc_auc,human_recall,ai_recall
model,,,,,,,
custom_cnn,0.969811,0.982143,0.948276,0.964912,0.996818,0.986577,0.948276
resnet18,0.932075,0.865672,1.000000,0.928000,0.998091,0.879195,1.000000
efficientnet_b0,0.905660,0.832117,0.982759,0.901186,0.984436,0.845638,0.982759


In [14]:
def evaluate_by_generator(model, metadata_df, transform):
    rows = []

    test_df = metadata_df[metadata_df["split"] == "test"]

    test_generators = test_df["generator"].unique()
    ai_generators = [g for g in test_generators if g != "Human"]

    human_n = (test_df["generator"] == "Human").sum()

    for gen in ai_generators:

        ai_n = (test_df["generator"] == gen).sum()

        sub_ds = SignatureDataset(
            metadata_df,
            "test",
            transform=transform,
            generators=["Human", gen]
        )

        if len(sub_ds) == 0:
            continue

        sub_loader = DataLoader(
            sub_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS
        )

        metrics, _, _, _ = evaluate_model(model, sub_loader)

        metrics["generator"] = gen
        metrics["human_n"] = human_n
        metrics["ai_n"] = ai_n
        metrics["n"] = human_n + ai_n

        rows.append(metrics)

    cols = [
        "generator",
        "human_n",
        "ai_n",
        "n",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "human_recall",
        "ai_recall"
    ]

    return pd.DataFrame(rows)[cols] if rows else pd.DataFrame(columns=cols)


generator_results = {}

for name, model in trained_models.items():

    print(f"--- {name}: per-generator test metrics (Human vs. each generator) ---")

    gen_df = evaluate_by_generator(
        model,
        metadata_df,
        eval_transform
    )

    generator_results[name] = gen_df

    display(gen_df.round(4))

--- custom_cnn: per-generator test metrics (Human vs. each generator) ---


,generator,human_n,ai_n,n,accuracy,precision,recall,f1,roc_auc,human_recall,ai_recall
0,Gemini,149,27,176,0.9545,0.9130,0.7778,0.8400,0.9863,0.9866,0.7778
1,QN,149,60,209,0.9904,0.9677,1.0000,0.9836,1.0000,0.9866,1.0000
2,GPT,149,29,178,0.9888,0.9355,1.0000,0.9667,1.0000,0.9866,1.0000


--- resnet18: per-generator test metrics (Human vs. each generator) ---


,generator,human_n,ai_n,n,accuracy,precision,recall,f1,roc_auc,human_recall,ai_recall
0,Gemini,149,27,176,0.8977,0.6000,1.0,0.7500,0.9928,0.8792,1.0
1,QN,149,60,209,0.9139,0.7692,1.0,0.8696,0.9996,0.8792,1.0
2,GPT,149,29,178,0.8989,0.6170,1.0,0.7632,1.0000,0.8792,1.0


--- efficientnet_b0: per-generator test metrics (Human vs. each generator) ---


,generator,human_n,ai_n,n,accuracy,precision,recall,f1,roc_auc,human_recall,ai_recall
0,Gemini,149,27,176,0.8580,0.5208,0.9259,0.6667,0.9575,0.8456,0.9259
1,QN,149,60,209,0.8900,0.7229,1.0000,0.8392,0.9894,0.8456,1.0000
2,GPT,149,29,178,0.8708,0.5577,1.0000,0.7160,0.9993,0.8456,1.0000
